In [1]:
from datasets import load_dataset
import pandas as pd
from collections import defaultdict
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import json
import re

import random
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import os
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from collections import Counter

# Bước chuẩn bị

### Load dataset train/test/valid

In [2]:
# Load dataset
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 56445
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
    test: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
})


In [3]:
# Chia tập train/test/valid
# Note: df_valid dùng để thử các kết quả model trước khi áp dụng lần cuối với test
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])
df_valid = pd.DataFrame(dataset['validation'])

In [4]:
df_train

,input,target
0,zinredie zidne: htlv có lương befo nhkấtl lịc ...,zinedine zidane: hlv có lương bèo nhất lịch sử...
1,Còn cứ kéo dài như vậy sẽ ko ổn.,Còn cứ kéo dài như vậy sẽ không ổn.
2,PVC trien khai nhieu cong trinh xay dung lon.,PVC triển khai nhiều công trình xây dựng lớn.
3,"""bữa cơm thwuwowrg khnôg có gì cao nơưn mỹ vị,...","""bữa cơm thưởng không có gì cao lương mỹ vị, c..."
4,"Dieu tra vu vo hui hang ti dong o Nam Can, Ca ...","Điều tra vụ vỡ hụi hàng tỉ đồng ở Năm Căn, Cà ..."
...,...,...
56440,May mà được ông làm tóc có tâm. Đến tết mái có...,May mà được ông làm tóc có tâm. Đến tết mái có...
56441,Nguoi dep Philippines dang quang HH The gioi 2...,Người đẹp Philippines đăng quang HH Thế giới 2...
56442,smartfone - người bn đồng hànk.,smartphone - người bạn đồng hành.
56443,"trongv vụ xử edan, cknúg ta cos lúg túng....","trong vụ xử vedan, chúng ta có lúng túng...."


### load and create vocabulary

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Tạo 1 vocab dựa trên 1 dataset khác
vocab = []
with open(r"/content/drive/MyDrive/vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx và ngược lại
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [4]:
# Tạo 1 vocab dựa trên 1 dataset khác
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx và ngược lại
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [6]:
print(len(vocab))
print(vocab[:10])

77814
['a', 'abagiua', 'abatoa', 'a_bàng', 'a_bung', 'a_di', 'a_di_đà_kinh', 'adiđà_phật', 'a_di_đà_phật', 'a_di_đà_tam_tôn']


Tạo 1 Dictionary là counts để tính số lần xuất hiện của mỗi từ trong vocab\
Lợi ích: sau này ta sẽ cộng điểm ưu tiên cho thằng có số lần xuất hiện lớn hơn\

In [5]:
counts = defaultdict(int)

for sentence in df_train['target']:
    sentence = sentence.lower()
    words = sentence.split()

    for i in range(len(words)):
        word = words[i]
        counts[word] += 1

# Các thuật toán sửa lỗi sẽ sử dụng:
(Note: thuật toán sửa lỗi 1, 2 là ở trong Symspell Correction, mình tách ra code riêng chứ k xài hàm có sẵn)

### Thuật toán 1: Sửa lỗi chính tả của từ
Sử dụng Symmetric Delete:
Chỉ sử dụng phép Xóa (Delete) để thu hẹp không gian tìm kiếm.\
Nếu hai từ có khoảng cách chỉnh sửa (Edit Distance) là $k$, chúng sẽ có chung ít nhất một biến thể "xóa đi $n$ ký tự" ($n \le k$).

Import file từ 1 chữ tiếng Việt -> nguyên âm - phụ âm - dấu câu trong telex\
VD: 'ắ' - 'a', 'w', 's'

In [6]:
with open(r"D:\NLP\project\telex.txt", "r", encoding="utf-8") as f:
    telex = f.read()

telex = re.sub(r',\s*}', '\n}', telex)
telex = json.loads(telex)
telex

{'ă': ['a', 'w', ''],
 'â': ['a', 'a', ''],
 'ê': ['e', 'e', ''],
 'ô': ['o', 'o', ''],
 'ơ': ['o', 'w', ''],
 'ư': ['u', 'w', ''],
 'đ': ['d', 'd', ''],
 'á': ['a', '', 's'],
 'à': ['a', '', 'f'],
 'ã': ['a', '', 'x'],
 'ả': ['a', '', 'r'],
 'ạ': ['a', '', 'j'],
 'ắ': ['a', 'w', 's'],
 'ằ': ['a', 'w', 'f'],
 'ẵ': ['a', 'w', 'x'],
 'ẳ': ['a', 'w', 'r'],
 'ặ': ['a', 'w', 'j'],
 'ấ': ['a', 'a', 's'],
 'ầ': ['a', 'a', 'f'],
 'ẫ': ['a', 'a', 'x'],
 'ẩ': ['a', 'a', 'r'],
 'ậ': ['a', 'a', 'j'],
 'é': ['e', '', 's'],
 'è': ['e', '', 'f'],
 'ẽ': ['e', '', 'x'],
 'ẻ': ['e', '', 'r'],
 'ẹ': ['e', '', 'j'],
 'ế': ['e', 'e', 's'],
 'ề': ['e', 'e', 'f'],
 'ễ': ['e', 'e', 'x'],
 'ể': ['e', 'e', 'r'],
 'ệ': ['e', 'e', 'j'],
 'í': ['i', '', 's'],
 'ì': ['i', '', 'f'],
 'ĩ': ['i', '', 'x'],
 'ỉ': ['i', '', 'r'],
 'ị': ['i', '', 'j'],
 'ó': ['o', '', 's'],
 'ò': ['o', '', 'f'],
 'õ': ['o', '', 'x'],
 'ỏ': ['o', '', 'r'],
 'ọ': ['o', '', 'j'],
 'ố': ['o', 'o', 's'],
 'ồ': ['o', 'o', 'f'],
 'ỗ': ['o', 'o'

Thử nghiệm hàm

In [7]:
def create_telex_form(word):
    word = word.lower()
    prefix = ""      # Phụ âm đầu
    vowel_base = ""  # Nguyên âm gốc
    suffix = ""      # Phụ âm cuối
    word_tone = ""   # Dấu thanh
    word_mod = ""    # Ký tự gõ mũ/móc
    
    VOWELS = "aeiouy" # Các nguyên âm tiếng Việt
    state = 0 # 0: phụ âm đầu, 1: nguyên âm
    
    i = 0
    while i < len(word):
        step = 1
        # Trường hợp 'ươ'
        if i < len(word) - 1 and word[i:i+2] in telex:
            char = word[i:i+2]
            step = 2
        else:
            char = word[i]
            
        # Nếu ký tự nằm trong từ điển
        if char in telex:
            if char == 'đ':
                if state == 0: prefix += 'dd'
                else: suffix += 'dd'
            else:
                vowel_base += telex[char][0]
                if telex[char][1]: word_mod = telex[char][1]
                if telex[char][2]: word_tone = telex[char][2]
                state = 1
                
        # Nếu ký tự là chữ bình thường
        else:
            if char in VOWELS:
                vowel_base += char
                state = 1
            else:
                if state == 0: prefix += char
                else: suffix += char
                
        i += step

    # Tạo biến thể
    variants = set()
    inline_vowel = vowel_base + word_mod
    
    # Kiểu 1 & 2: Gõ chuẩn ngay sau nguyên âm hoặc ném dấu thanh ra cuối
    variants.add(prefix + inline_vowel + word_tone + suffix)
    variants.add(prefix + inline_vowel + suffix + word_tone)
    
    # Kiểu 3: Phím bổ nghĩa (w, a, e, o) ném ra cuối từ
    if word_mod:
        variants.add(prefix + vowel_base + word_tone + suffix + word_mod)
        variants.add(prefix + vowel_base + suffix + word_mod + word_tone)
        variants.add(prefix + vowel_base + suffix + word_tone + word_mod)
        
    # Trường hợp đặc biệt của "ươ"
    if vowel_base == 'uo' and word_mod == 'w':
        # Tách w hai lần ngay sau nguyên âm
        variants.add(prefix + 'uwow' + word_tone + suffix)
        variants.add(prefix + 'uwow' + suffix + word_tone)
        
        # Tách w đầu, w cuối
        variants.add(prefix + 'uwo' + word_tone + suffix + 'w')
        variants.add(prefix + 'uwo' + suffix + 'w' + word_tone)
        variants.add(prefix + 'uwo' + suffix + word_tone + 'w')
        
    return list(v for v in variants if v)

In [43]:
create_telex_form("trường")

['truwongwf',
 'truwowngf',
 'truwofngw',
 'truwongfw',
 'truofngw',
 'truongwf',
 'truongfw',
 'truwowfng',
 'truowfng',
 'truowngf']

Hàm tạo các biến thể xóa từ 0 đến k kí tự của từ, hàm trả về list các biến thể xóa

In [44]:
def get_deletes(word, k = 2):
    queue = [word]
    variant_list = set()

    for _ in range(k):
        temp_queue = set()
        for w in queue:
            if len(w) > 1:
                for i in range(len(w)):
                    variant = w[:i] + w[i+1:]
                    variant_list.add(variant)
                    temp_queue.add(variant)
        queue = temp_queue
    return variant_list

In [45]:
# Kiểm tra hàm
print(get_deletes("chào"))

{'cào', 'ch', 'hà', 'co', 'cà', 'ho', 'ào', 'chà', 'cho', 'hào'}


In [112]:
# Tạo danh sách các từ thiếu từ 0 đến k kí tự trong vocab
# Tạo 1 Dictionary (Hash Map) để lưu toàn bộ biến thể xóa
sym_dict = defaultdict(list)

for word in vocab:
    # Lấy các biến thể và từ gốc của 1 từ
    base_forms = [word] + create_telex_form(word)

    for form in base_forms:
        # Lưu form này (distance 0)
        if word not in sym_dict[form]:
            sym_dict[form].append(word)
        
        # Tạo deletes cho từng form và lưu vào từ điển
        variant_list = get_deletes(form)
        for variant in variant_list:
            # Map biến thể tới từ gốc 'word' hiện tại nếu chưa map
            if word not in sym_dict[variant]:
                sym_dict[variant].append(word)

In [47]:
# Kiểm tra
sym_dict

defaultdict(list,
            {'a': ['a',
              'aga',
              'ala',
              'ami',
              'avi',
              'à',
              'ả',
              'á',
              'ạ',
              'abc',
              'ác',
              'adn',
              'ag',
              'ai',
              'ải',
              'ái',
              'ak',
              'al',
              'am',
              'ám',
              'an',
              'án',
              'ang',
              'anh',
              'ao',
              'ào',
              'ảo',
              'áo',
              'áp',
              'ar',
              'as',
              'át',
              'atk',
              'au',
              'áy',
              'ăă',
              'ă',
              'ăn',
              'â',
              'âm',
              'ân',
              'âu',
              'ba',
              'bà',
              'bả',
              'bã',
              'bá',
              'bạ',
              '

In [115]:
# Hàm tính khoảng cách của xâu 1 và xâu 2 bằng thuật toán Damerau-Levenshtein
# Là hàm edit_distance nhưng có thêm phép đổi chỗ các kí tự (gõ lộn thứ tự)
# Cải thiện thêm bằng khoảng cách bàn phím cho trường hợp gõ nhầm

# Các phím liền kề trên bàn phím để tính trọng số
ADJACENT_KEYS = {
    'q': 'wea', 'w': 'qeasd', 'e': 'wrsdf', 'r': 'etdfg', 't': 'ryfgh', 'y': 'tughj', 'u': 'yihjk', 'i': 'uojkl', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwsz', 's': 'weadzx', 'd': 'ersfxc', 'f': 'rtdgcv', 'g': 'tyfhvb', 'h': 'yugjbn', 'j': 'uihknm', 'k': 'iojlm', 'l': 'opk',
    'z': 'asx', 'x': 'sdzc', 'c': 'dfxv', 'v': 'fgcb', 'b': 'ghvn', 'n': 'hjbm', 'm': 'jkn'
}

# Các phím mang tính năng của Telex
TELEX_KEYS = set(['s', 'f', 'r', 'x', 'j', 'w', 'a', 'e', 'o', 'd'])

def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]

    # Khởi tạo giá trị hàng và cột đầu tiên
    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            char1 = s1[i - 1]
            char2 = s2[j - 1]
            # Tính chi phí thay thế (0 nếu giống nhau, 0.5 nếu khác mà gần nhau trên bàn phím, 1 nếu khác và ở xa trên bàn phím)
            if char1 == char2:
                sub_cost = 0.0
            elif char1 in ADJACENT_KEYS.get(char2, "") or char2 in ADJACENT_KEYS.get(char1, ""):
                # Nếu gõ nhầm 2 phím cạnh nhau (VD: a và s), chi phí chỉ là 0.5
                sub_cost = 0.5 
            else:
                # Lỗi gõ nhầm phím xa nhau, chi phí 1.0
                sub_cost = 1.0

            # Tính chi phí thêm / xóa
            del_cost = 0.8 if char1 in TELEX_KEYS else 1.0
            ins_cost = 0.8 if char2 in TELEX_KEYS else 1.0

            dp[i][j] = min(
                dp[i - 1][j] + del_cost,       # Xóa
                dp[i][j - 1] + ins_cost,       # Thêm
                dp[i - 1][j - 1] + sub_cost    # Thay thế
            )

            # Phép Đổi chỗ (Transposition)
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 0.5)

    return dp[n][m]

In [116]:
# Kiểm tra hàm
edit_distance("truwowfng", "trwuwowfng")

0.8

In [117]:
# Hàm tìm từ gần nhất với từ nhập vào, hàm trả về các từ gần nhất và khoảng cách của nó
def lookup(word, k=2):
    # Trả về tuple (từ, khoảng cách) để đồng nhất với kết quả ở dưới
    if word in vocab:
        return word

    variant_list = get_deletes(word)
    variant_list.add(word)

    # Lưu đáp án là các từ gần nhất và khoảng cách của nó
    candidates = {}

    # Tra cứu các biến thể
    for variant in variant_list:
        if variant in sym_dict:
            for suggestion in sym_dict[variant]:
                if suggestion in candidates:
                    continue

                # SỬ DỤNG HÀM TELEX ĐỂ TÍNH KHOẢNG CÁCH (NẾU BẠN KIỂM TRA LỖI GÕ PHÍM)
                telex_form = [suggestion] + create_telex_form(suggestion)

                dist = min([edit_distance(word, form) for form in telex_form])

                # Nhận khi khoảng cách <= k, tồn tại trong từ điển và có tần suất > 0
                if dist <= k and suggestion in vocab and counts.get(suggestion, 0) > 0:
                    candidates[suggestion] = (dist, counts.get(suggestion, 0))

    # Xếp hạng ưu tiên: Distance nhỏ trước -> Tần suất cao trước
    result = sorted(candidates.items(),
                    key=lambda x: (x[1][0], -x[1][1]))

    ans = []
    # Giải nén thẳng tuple cho dễ đọc, đổi tên biến tránh ghi đè tham số 'word'
    for cand_word, (dist, count) in result:
        # Sử dụng tham số k thay vì hardcode số 2
        if dist > k: 
            break
        
        # Trả về cả từ và khoảng cách 
        ans.append(cand_word)
        
    return ans

In [75]:
# Kiểm tra hàm
lookup("trwuwowfng")

['trường', 'trưởng', 'trương', 'tường', 'rường']

### Thuật toán 2: viết tắt (Teencode & Informal Variants)
Mapping (tạo các cặp viết tắt và từ chính) + Contextual Ranking (xem thử cái nào khả thi hơn)\
Kết hợp Deep Learning để xem xét ngữ cảnh của từ viết tắt từ đó tăng độ chính xác

### Thuật toán 3: Sửa lỗi viết không dấu
2 cách:
- 1 là N-gram + Viterbi
- 2 là model BARTpho-syllable nhỏ

### Model chung: Chọn từ phù hợp với ngữ cảnh
Dựa trên Skip-gram (window_size = 5) + log-Probability\
Hướng sử dụng là check tìm trong câu các từ không có trong từ điển -> gán lỗi cho nó -> sử dụng thuật toán sửa lỗi 1 tìm ra top n từ có thể sẽ đúng (ở đây n = 5) -> xem thử có trong từ điển không -> xem từ nào gần nhất với các từ trong câu\
VD: hôm nay tôi đi hõc ở trường
- hõc không có trong từ điển -> sai -> các từ thay thế: học, hóc, húc, hạc, hắc
- tính điểm dựa trên các yếu tố sau và chọn thằng có điểm cao nhất
    - 1. Điểm dựa vào ngữ cảnh (skip-gram trừ stopwords)
    - 2. Điểm dựa vào tần xuất (n-gram)
    - 3. Điểm cộng nếu tạo được thành 1 từ có trong từ điển
    - 4. Điểm trừ nếu ứng cử viên là 1 từ trong stopword

Load danh sách stopwords

In [12]:
with open(r"D:\NLP\project\vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

Tạo các cặp skipgram với window_size = 5

In [13]:
window_size = 5
skipgram_data = []

for sentence in df_train['target']:
    # Lấy danh sách từ
    sentence = sentence.lower()
    words = sentence.split()

    # Lọc stopword
    filtered_words = []
    for word in words:
        if word not in word_to_idx or word in stopword:
            continue

        filtered_words.append(word)

    for i, word in enumerate(filtered_words):
        # Kiểm tra xem từ này có trong vocab
        if word not in word_to_idx:
            continue

        center = word_to_idx[word]

        for j in range(-window_size, window_size + 1):
            # Kiểm tra các khoảng cách xem có phù hợp (trừ chính nó)
            if j == 0:
                continue
            if 0 <= i + j < len(filtered_words):
                context_word = words[i + j]
                #Kiểm tra các từ bên cạnh có trong vocab không
                if context_word in word_to_idx:
                    context = word_to_idx[context_word]
                    skipgram_data.append((center, context))

In [26]:
len(skipgram_data)

1996340

model Skipgram

In [14]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        embed = self.embedding(x)
        out = self.linear(embed)

        return out

In [ ]:
#  Chuẩn bị chạy bằng GPU
device = torch.device("cuda")

centers = torch.LongTensor([pair[0] for pair in skipgram_data])
contexts = torch.LongTensor([pair[1] for pair in skipgram_data])

batch_size = 4096
dataset = TensorDataset(centers, contexts)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
EMBED_DIM = 256
VOCAB_SIZE = len(vocab)
model_skipgram = SkipGram(VOCAB_SIZE, EMBED_DIM).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_skipgram.parameters(), lr=0.001)

epochs = 30

model_skipgram.train()

for epoch in range(epochs):
    total_loss = 0

    for batch_centers, batch_contexts in loader:

        batch_centers = batch_centers.to(device)
        batch_contexts = batch_contexts.to(device)

        # Forward
        outputs = model_skipgram(batch_centers)
        loss = criterion(outputs, batch_contexts)

        # Backward
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 3827.7262
Epoch 2, Loss: 3245.9217
Epoch 3, Loss: 3199.7056
Epoch 4, Loss: 3174.3389
Epoch 5, Loss: 3157.1457
Epoch 6, Loss: 3144.0318
Epoch 7, Loss: 3133.5485
Epoch 8, Loss: 3124.6086
Epoch 9, Loss: 3116.9301
Epoch 10, Loss: 3110.2487
Epoch 11, Loss: 3104.1077
Epoch 12, Loss: 3098.6840
Epoch 13, Loss: 3093.6828
Epoch 14, Loss: 3089.0322
Epoch 15, Loss: 3084.6882
Epoch 16, Loss: 3080.6714
Epoch 17, Loss: 3076.9367
Epoch 18, Loss: 3073.3004
Epoch 19, Loss: 3070.0118
Epoch 20, Loss: 3066.7181
Epoch 21, Loss: 3063.7396
Epoch 22, Loss: 3060.7959
Epoch 23, Loss: 3058.0509
Epoch 24, Loss: 3055.3211
Epoch 25, Loss: 3052.7774
Epoch 26, Loss: 3050.3782
Epoch 27, Loss: 3048.0014
Epoch 28, Loss: 3045.6947
Epoch 29, Loss: 3043.4060
Epoch 30, Loss: 3041.3871


Lưu model train skip-gram

In [ ]:
torch.save(model_skipgram, r'D:/NLP/project/model_skipgram.pth')

In [15]:
model_skipgram = torch.load(r'D:/NLP/project/model_skipgram.pth', weights_only=False)

In [ ]:
torch.save(model_skipgram, 'model_skipgram.pth')
from google.colab import files

files.download('model_skipgram.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hàm tính độ liên quan của 2 từ

In [25]:
def calculate_similarity(word1, word2, embedding_matrix, word_to_idx):
    # Lấy vector của từng từ
    idx1 = word_to_idx[word1]
    idx2 = word_to_idx[word2]
    vec1 = embedding_matrix[idx1]
    vec2 = embedding_matrix[idx2]

    # Tính Cosine Similarity
    # Công thức: (A . B) / (||A|| * ||B||)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

Test thử với các từ "hóc xương", "học xương". "hóc xương" sẽ phải cho ra điểm cao hơn so với "học xương"

In [17]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

score1 = calculate_similarity("hóc", "xương", embedding_matrix, word_to_idx)
score2 = calculate_similarity("học", "xương", embedding_matrix, word_to_idx)

print(score1, score2)

0.06637567 0.019825403


Test thử với các từ "sóng thần", "sóng tiền". "sóng thần" sẽ phải cho ra điểm cao hơn so với "sóng tiền"

In [18]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

score1 = calculate_similarity("sóng", "thần", embedding_matrix, word_to_idx)
score2 = calculate_similarity("sóng", "tiền", embedding_matrix, word_to_idx)

print(score1, score2)

0.14927007 0.05731002


Chuẩn bị để chuẩn hóa điểm count về [0, 1] cho dễ tính toán

In [19]:
all_log_counts = [math.log(c) for c in counts.values()]
min_log = min(all_log_counts)
max_log = max(all_log_counts)

def get_normalized_ngram(word, counts, min_log, max_log):
    c = counts.get(word, 1)
    log_c = math.log(c)
    # Ép về khoảng [0, 1]
    return (log_c - min_log) / (max_log - min_log + 1e-9)

Hàm chọn từ sửa sai dựa trên độ liên quan

In [136]:
def choose_best_candidate(candidates, sentence, error_idx, embedding_matrix, error_indices, 
                           word_to_idx, stopwords, counts, min_log, max_log, window_size=3):
    best_word = None
    max_total_score = -float('inf')

    # Chọn các từ để candidate xét, bỏ chính nó, các từ stopwords
    start = max(0, error_idx - window_size)
    end = min(len(sentence), error_idx + window_size + 1)
    
    context_words = [sentence[i] for i in range(start, end) 
                     if i not in error_indices and sentence[i] not in stopwords]

    for candidate in candidates:
        # 1. Tính điểm dựa trên ngữ cảnh (chuẩn hóa lại điểm từ 0 đến 1)
        sim_sum = 0
        valid_contexts = 0
        for context in context_words:
            if context in word_to_idx:
                sim = calculate_similarity(candidate, context, embedding_matrix, word_to_idx)
                sim_sum += max(0, sim) # Chỉ lấy điểm dương
                valid_contexts += 1
        
        avg_sim = sim_sum / valid_contexts if valid_contexts > 0 else 0.0

        # 2. Tính điểm tần suất (chuẩn hóa lại điểm từ 0 đến 1)
        norm_ngram = get_normalized_ngram(candidate, counts, min_log, max_log)

        # 3. Kiểm tra xem có tạo từ ghép tồn tại trong từ điển (điểm thưởng)
        bigram_bonus = 0.0
        
        # Kiểm tra với từ đứng trước
        if error_idx > 0:
            prev_word = sentence[error_idx - 1]
            compound_prev = f"{prev_word}_{candidate}"
            if compound_prev in word_to_idx:
                bigram_bonus += 2.0 # Cộng một điểm thưởng rất lớn

        # Kiểm tra với từ đứng sau
        if error_idx < len(sentence) - 1:
            next_word = sentence[error_idx + 1]
            compound_next = f"{candidate}_{next_word}"
            if compound_next in word_to_idx:
                bigram_bonus += 2.0
        
        # 4. Trừ điểm nếu candidate là stopword
        penalty = 0.5 if candidate in stopwords else 0.0

        # Trộn điểm với trọng số
        # Alpha=0.7 (Tin vào ngữ cảnh), Beta=0.3 (Tin vào tần suất)
        total_score = (0.5 * avg_sim) + (0.2 * norm_ngram) + (0.3 * bigram_bonus) - penalty

        if total_score > max_total_score:
            max_total_score = total_score
            best_word = candidate
        print(candidate, total_score)

    return best_word if best_word is not None else candidates[0]

Tạo hàm để thực hiện thuật toán

In [80]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s_]', '', text)
    text = text.split()
    return text

In [81]:
def correct_spelling_errors(sentence, embedding_matrix, word_to_idx, stopwords, counts):
    sentence = preprocess(sentence)
    error = []

    # Tìm vị trí các từ bị lỗi
    for i, word in enumerate(sentence):
        if word in word_to_idx:
            continue

        candidates = lookup(word)
        if len(candidates) == 0:
            continue
        
        error.append(i)
        
    # Bắt đầu quá trình sửa lỗi
    for error_idx in error:
        candidates = lookup(sentence[error_idx])
        
        # Gọi hàm và truyền đầy đủ các tham số cần thiết
        if candidates:
            sentence[error_idx] = choose_best_candidate(
                candidates=candidates, 
                sentence=sentence, 
                error_idx=error_idx, 
                embedding_matrix=embedding_matrix, 
                error_indices=error,
                word_to_idx=word_to_idx,
                stopwords=stopwords,
                counts=counts,
                min_log=min_log,
                max_log=max_log
            )

    # Dùng join để ghép chuỗi chuẩn xác và tối ưu hiệu năng
    sentence = " ".join(sentence)
    return sentence

In [95]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

# Test sửa lỗi
sentence = "Hôm nay tôi đi hõc ở trường."
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

học 0.7713953
hóc 0.07176587
húc 0.087011546
hạc 0.07243934
hục 0.045936234
hà 0.18796875
hlv 0.15537778
hè 0.1250555
úc 0.121016935
hò 0.1015225
hì 0.06203063
hiv 0.069493786
htx 0.041999906
hù 0.041783683
hũ 0.046416245
bbc 0.0699504
hd 0.05640897
hay -0.3386532407801019
hai 0.17075938
hoa 0.1628281
họ -0.34083414
hả 0.12939323
hạ 0.12290796335994073
ha 0.15407336
ác 0.11520971
hư 0.10351751
óc 0.120005414
hô 0.09434678
mc 0.104433976
heo 0.1112507
ho 0.09095138
hí 0.13694668
hú 0.09213648
ham 0.14182049
hé 0.10671383
há 0.083040886
c 0.068359055
he 0.0850833
hao 0.09106706
hen 0.062489524
pc 0.112534374
han 0.05503314
h 0.07722646
hủ 0.07345544
đc 0.051211465
hoi 0.0788647
hun 0.105514124
hh 0.054147664
cc 0.09143095
hỉ 0.07095617
hiu 0.06756085
hơ 0.06779989
hê 0.047845438
him 0.049422868
hỷ 0.054361742
hẹ 0.046040032
hom 0.02268178
ht 0.038582996
hụ 0.032943875
sic 0.0314593
ic 0.04227718
cnc 0.038011547
hg 0.018365556
hia 0.036287244
hoe 0.0047861505
hôm nay tôi đi học ở trường


In [130]:
sentence = "sóng thền thì dất nguy hiểm"
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

thềm 0.068620495
tiền 0.17594011
thân 0.18598187
thần 0.7920752
thận 0.12711178
thôn 0.113013156
thề 0.11671731
than 0.10863886
thon 0.08136311
then 0.08566062
thốn 0.048850466
thẹn 0.062050268
thìn 0.08637173
thiền 0.044148907
thồn 0.026434936
thun 0.027516294
thị 0.14754573
thêm -0.30907398
tham 0.13605657
tuần 0.13734457
thăm 0.12396815
nền -0.36180404
tuấn 0.109008335
thẩm 0.12228601
thảm 0.15812981
thụ 0.13422632
bền 0.12749253
hiền 0.11313068
thơm 0.10719894
thậm -0.3962432
thèm 0.10994308
tuân 0.12866458
thâm 0.1528272
thọ 0.12047039
thắm 0.09235966
tiềm 0.15046136
ghen 0.0761880760662438
tuyền 0.08494126
thầm 0.7020104
thấm 0.10171996
thám 0.07802509
thím -0.451697902573165
ghiền 0.07695144
ghềnh 0.057286076
tuồn 0.07290641
tuôn 0.036188994
thẳm 0.0521196
thẫm 0.030436866
thỏm -0.48476238478675127
triền 0.049329855
thì -0.2937466
thế 0.26716920841889136
trên -0.3343177
thành 0.17512049
thấy -0.31415924
thể 0.22378743
nhân 0.19315882
hơn -0.33906716
theo -0.3241431
thôi 0.254580

In [137]:
sentence = "ít nhất 5 người chết do sóng thnầ tạj xloomon."
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

thần 0.79727125
tuần 0.1343894
trần 0.16883425
phần -0.35849452
tung 0.152744
tần 0.07530624
thuần -0.39438316
chần 0.063147485
thanh -0.35143524128590886
thân 0.16382197
thôn 0.10669599
thang 0.09811273
than 0.10007724
thon 0.10908161
then 0.07518777
thẹn 0.059605055
thì -0.2937466
thế -0.3276596
thành 0.19701824
thấy -0.3181215
thể 0.1902866
theo -0.32322058
thôi -0.33599105
tin -0.33347198
tăng -0.3061876
thời 0.15120784585787306
tình 0.76329535
thực 0.16086577
thật -0.3394395
thông 0.16000624
thủ 0.17024496
thị 0.15480883
thi 0.15463597
thằng 0.172299
thầy 0.18180779
tháng -0.31437564
thu 0.7861064
thức 0.1814494
thứ -0.34828463
thêm -0.32151014
tính -0.36148836470398193
tổng 0.15920895
thống 0.1874773
tham 0.13659158
tên -0.34162343
thái 0.15360291
tỉnh 0.16446444
thay 0.16205797
thắng 0.16543064
thư 0.16353813
từng -0.354003
tấn -0.3770101623111424
thịt 0.13965558
thử 0.13851753
tinh 0.1305174
thảo 0.1200193344523905
tặng 0.1365358
thất 0.16049609
tùng 0.13598728
thí 0.16340357
t

# Kết hợp các thuật toán sửa lỗi
Biến các thuật toán thành 1 hàm sửa lỗi câu theo trường hợp
- 1. Sai lỗi chính tả  
- 2. Viết tắt
- 3. Viết không dấu     \
=> Tất cả sẽ dựa trên ngữ cảnh

# Test trên tập valid

Lấy top 1 câu sau chỉnh sửa để xem trên tập valid xem kết quả

# Chỉnh sửa phù hợp

Điều chỉnh thuật toán, dữ liệu, vv cho phù hợp lại

# Đánh giá lần cuối trên tập test